In [0]:
%sql
use catalog identifier(:catalog);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

CREATE OR REPLACE TABLE gold.claims_enriched AS
SELECT
  c.claim_id, c.member_id, c.policy_id, c.hospital_id,
  c.admission_date, c.discharge_date,
  cast(date_trunc('month', c.admission_date) AS DATE) AS admission_month,
  c.diagnosis_code, c.procedure_code,
  c.is_pre_existing, c.claim_type,
  c.billed_amount, c.room_rent_per_day, c.room_days,
  c.status, c.approved_amount, c.rejection_code,
  c.submitted_ts, c.settled_ts, c.channel,
  -- plan facts in force on the admission date (SCD2 as-of join)
  p.plan_code AS plan_code_at_admission,
  pl.plan_family, pl.room_rent_cap_per_day,
  p.effective_from AS policy_start_date,
  cast(months_between(c.admission_date, p.effective_from) AS INT) AS policy_tenure_months,
  -- claim outcome facts
  datediff(c.discharge_date, c.admission_date) AS length_of_stay,
  coalesce(c.room_rent_per_day > pl.room_rent_cap_per_day, false) AS room_rent_over_cap,
  datediff(cast(c.settled_ts AS DATE), cast(c.submitted_ts AS DATE)) AS turnaround_days,
  -- network fact on the admission date (never empty)
  coalesce(h.hospital_id IS NOT NULL, false) AS is_network_at_admission,
  -- hospital facts
  hs.city AS hospital_city,
  hs.city_tier AS hospital_city_tier
FROM silver.claims_clean c
LEFT JOIN silver.dim_policy_scd2 p
  ON p.member_id = c.member_id
 AND p.effective_from <= c.admission_date
 AND (p.effective_till IS NULL OR c.admission_date <= p.effective_till)
LEFT JOIN raw.plans pl
  ON pl.plan_code = p.plan_code
LEFT JOIN raw.hospital_network_history h
  ON h.hospital_id = c.hospital_id
 AND h.network_from <= c.admission_date
 AND (h.network_to IS NULL OR c.admission_date <= h.network_to)
LEFT JOIN raw.hospitals hs
  ON hs.hospital_id = c.hospital_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.hospital_monthly AS
SELECT
  hospital_id,
  admission_month,

  cast(count(*) AS INT)                    AS claim_count,
  cast(sum(billed_amount) AS DECIMAL(18,2))   AS total_billed,
  cast(sum(approved_amount) AS DECIMAL(18,2)) AS total_approved,
  cast(count_if(status = 'rejected') AS INT)  AS rejected_count
FROM gold.claims_enriched
GROUP BY hospital_id, admission_month;

In [0]:
%sql
SELECT 'fact_rows' AS check_name, cast(count(*) AS double) AS value FROM gold.claims_enriched
UNION ALL
SELECT 'summary_claim_count_total', cast(sum(claim_count) AS double) FROM gold.hospital_monthly
UNION ALL
SELECT 'fact_rejected_total', cast(count_if(status = 'rejected') AS double) FROM gold.claims_enriched
UNION ALL
SELECT 'summary_rejected_count_total', cast(sum(rejected_count) AS double) FROM gold.hospital_monthly
UNION ALL
SELECT 'fact_billed_total', sum(billed_amount) FROM gold.claims_enriched
UNION ALL
SELECT 'summary_total_billed', sum(total_billed) FROM gold.hospital_monthly
UNION ALL
SELECT 'fact_approved_total', sum(approved_amount) FROM gold.claims_enriched
UNION ALL
SELECT 'summary_total_approved', sum(total_approved) FROM gold.hospital_monthly

In [0]:
%sql
SELECT f.claim_id, f.member_id, f.admission_month,
       f.plan_code_at_admission, f.policy_start_date, f.policy_tenure_months,
       cur.plan_code AS plan_code_today
FROM gold.claims_enriched f
JOIN silver.dim_policy_scd2 cur
  ON cur.member_id = f.member_id AND cur.is_current
WHERE EXISTS (
  SELECT 1 FROM silver.dim_policy_scd2 prev
  WHERE prev.member_id = f.member_id
    AND prev.effective_till IS NOT NULL
    AND prev.effective_from <= f.admission_date
    AND (prev.effective_till IS NULL OR f.admission_date <= prev.effective_till))
LIMIT 5;



In [0]:
%sql
SELECT claim_id, hospital_id, admission_month, is_network_at_admission
FROM gold.claims_enriched
WHERE NOT is_network_at_admission
LIMIT 5;